# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary from metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("\nDataset identifier:", getattr(metadata,'identifier',None))
print("Version:", getattr(metadata,'version',None))
print("License:", getattr(metadata,'license',None))

# Show keywords
print("Keywords:", getattr(metadata,'keywords',None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data in record sets, each with fields and columns. We use `@id` references to access these entities.

In [ ]:
# List available record set @id's
record_sets = dataset.record_sets
print("Record Sets Found:")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | name: {rs.get('name','')}")

# Show fields for each record set
for rs in record_sets:
    print("\nRecord Set:", rs['@id'], f"({rs.get('name','')})")
    print("Fields:")
    for field in rs.get('fields', []):
        print(f"   @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")
    print("Columns:")
    for col in rs.get('columns', []):
        print(f"   @id: {col['@id']} | name: {col.get('name','')} | source: {col.get('source','')}")

## 3. Data Extraction
Load data from each record set into Pandas DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll demonstrate loading the first record set, and list its columns.

In [ ]:
# Extract data from all record sets using their @id
dataframes = {}
# Extract each record set's @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded DataFrame for Record Set @id: {rs_id}")
    print(df.columns.tolist())
    print(df.head())

# Choose the first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, or grouping by key attributes.

*Note: The field `@id`s are used for all references. We'll select a numeric column for demonstration. If numeric fields are not present, we'll use example columns as shown in the overview above.*

In [ ]:
import numpy as np

# Use the first record set loaded
df = dataframes[main_record_set_id]
# Find numeric columns by dtype or by field overview
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric columns:", numeric_cols)

if numeric_cols:
    numeric_field_id = numeric_cols[0]
else:
    numeric_field_id = df.columns[0] # fallback, e.g. 'age_at_diagnosis' if present

# Set threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field, e.g. use 'infertility' or similar
    possible_group_fields = [col for col in df.columns if col != numeric_field_id]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the selected numeric field, as well as its values grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id: {numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Grouped boxplot by group_field
    if possible_group_fields:
        plt.figure(figsize=(7,4))
        group_field_plot = possible_group_fields[0]
        sns.boxplot(x=group_field_plot, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_plot}")
        plt.xlabel(group_field_plot)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed genotype–phenotype and clinical tabulations for three female patients with non-classical 21-hydroxylase deficiency-associated infertility.
- Record sets, fields, and columns are referenced using their `@id`s, ensuring precise mapping to schema elements.
- Data loading and analysis was performed with the `mlcroissant` library, demonstrating field-level processing, normalization, and visualization.
- The dataset's small sample size and clinical specificity limit generalizability but are valuable for rare disease research and reproducibility.
- Further enrichment would require additional cases and structured harmonization.

For more information or schema element breakdown, refer to the Croissant schema at [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json).